FaVOR: LLM-Based Agentic Framework for Factor Mining via Empirical Validation
link : https://www.alphaxiv.org/pdf/2608.30192


## 한 줄 요약

LLM 팩터 마이닝이 수익률만 보고 최적화하다 보니 **수식이 원래 의도한 경제적 가설을 실제로 측정하는지**를 확인하지 않는다는 문제를 지적하고, 가설을 관찰 가능한 시장 상태로 쪼갠 뒤(분해) 각 팩터가 그 상태를 실제로 포착하는지 분포로 검증하고(검증) 통과한 팩터들을 조합하는(통합) 3단계 에이전트 프레임워크 FaVOR를 제안한 논문입니다. 2026년 8월 arXiv에 올라왔고, 동국대·서울대·숭실대 저자들입니다.

---

## 1. 문제의식

전통적인 팩터 연구는 시장 관찰 → 경제적 가설 → 팩터 수식 → 실증 검증 순서로 진행됩니다. 이 전통에서 팩터는 예측 공식이 아니라 특정 경제 메커니즘을 측정하는 도구입니다.

반면 FAMA, AlphaAgent, RD-Agent 같은 LLM 기반 방법들은 가설에서 수식으로 곧장 넘어가고, 그 수식을 백테스트 수익률로 평가합니다. 그러면 수식이 가설과 무관한 우연한 상관을 잡아도 수익률만 좋으면 통과합니다. 저자들은 이런 팩터가 표본 내에서는 좋아 보이지만 레짐이 바뀌면 무너진다고 주장합니다. 핵심은 **"수식의 형태"와 "경제적 의미" 사이의 불일치를 구조적 실패 원인으로 보는 것**입니다.

---

## 2. 방법론

### Stage 1: 가설 분해

세 개의 LLM 에이전트가 순서대로 작동합니다.

**가설 에이전트** $\mathcal{A}_H(I) = h$: 사람이 준 투자 아이디어 $I$(예: "하락 후 평균회귀")를 받아, 일봉 OHLCV만으로 관찰 가능한 구체적 시장 상태와 그 결과를 담은 가설 $h$를 1~3문장으로 씁니다. 보유 기간(1~10일)도 함께 정합니다. 논문 예시는 "급락 후 장중에 회복해 고가 근처에서 마감한 종목은 매수세 유입을 반영하며, 향후 3일간 단기 반등 확률이 높다"입니다.

**관찰 에이전트** $\mathcal{A}_O(h) = \{o_i \mid i = 1, \ldots, m\}$: 가설을 서로 독립적인 관찰 조건 $m$개로 쪼갭니다. 위 예시에서는 "급락 이후 상태", "장중 회복", "초기 안정화" 세 개입니다. 같은 현상을 다른 말로 반복하지 말 것, 최소 하나는 전환 신호를 포함할 것 같은 규칙이 프롬프트에 들어 있습니다.

**팩터 에이전트** $\mathcal{A}_F(o_i) = F_i = \{f_{i,j}\}_{j=1}^{n_i}$: 관찰 조건 하나당 서로 다른 접근의 후보 수식을 2~3개 만듭니다. 미리 정해진 연산자 목록(부록 A: RANK, TS_ZSCORE, DELTA, TS_CORR 등)만 쓸 수 있습니다. 각 수식에는 극성(polarity), 즉 "값이 클수록 관찰 조건이 더 참인지, 작을수록 더 참인지"가 지정됩니다.

### Stage 2: 팩터 수준 검증

이 논문의 핵심입니다. 중요한 점은 **미래 수익률을 쓰지 않는다**는 것입니다. 이 팩터가 수익을 내는지가 아니라, 이 팩터 값이 클수록 의도한 시장 상태가 실제로 더 강하게 나타나는지를 봅니다.

**절차**: 학습 기간 동안 날짜마다 팩터 값으로 종목을 정렬해 5분위 $Q_1, \ldots, Q_5$로 나눕니다. $Q_1$은 관찰 조건이 약한 쪽, $Q_5$는 강한 쪽입니다. 그다음 각 분위에 속한 종목들의 같은 날 시장 상태 변수 네 개의 분포를 봅니다.

$$\mathcal{X} = \{\text{DIR} = C - O,\ \ \text{MAG} = H - L,\ \ \text{POS} = \tfrac{C - L}{H - L},\ \ \text{VOL} = V\}$$

DIR은 장중 방향(양봉/음봉 크기), MAG는 하루 가격 범위(변동 크기), POS는 종가가 하루 범위의 어디에 위치하는지(0이면 저가 마감, 1이면 고가 마감), VOL은 거래량입니다. 분위별로 이 변수들의 평균, 중앙값, 10·90 분위수, 첨도 등을 계산해 통계 프로파일 $S_{i,j}$를 만듭니다.

**네 가지 검증 기준** (부록 B):

(i) 중심 경향 이동. 분위 $k$에서의 평균과 중앙값을

$$\mu_k(x) = \mathbb{E}[x_{s,t} \mid (s,t) \in Q_k],\quad m_k(x) = \text{Median}[x_{s,t} \mid (s,t) \in Q_k]$$

로 정의합니다. 인접한 분위끼리 평균의 변화 방향과 중앙값의 변화 방향이 일치해야 합니다.

$$\text{sign}(\mu_{k+1}(x) - \mu_k(x)) = \text{sign}(m_{k+1}(x) - m_k(x)),\quad k = 1, \ldots, l-1$$

여기에 분위 번호 $k$와 $\mu_k(x)$ 사이 상관이 0이 아니어야 한다는 조건이 붙습니다. 쉽게 말해, 팩터가 강해질수록 어떤 상태 변수가 한 방향으로 꾸준히 움직이고, 그 움직임이 평균에서도 중앙값에서도 같게 보여야 한다는 뜻입니다. 평균만 보면 극단값 하나에 끌려갈 수 있으니 중앙값으로 교차 확인하는 것입니다.

(ii) 꼬리 행동. 분위별 꼬리 폭을

$$\Delta^{\text{tail}}_k(x) = q_{0.9}(x \mid Q_k) - q_{0.1}(x \mid Q_k)$$

로 정의하고, 어떤 두 분위 $k_1 \ne k_2$에서 $|\Delta^{\text{tail}}_{k_1}(x) - \Delta^{\text{tail}}_{k_2}(x)| > \epsilon$이면 통과입니다. 팩터가 평균적인 종목뿐 아니라 분포의 양 끝까지 영향을 주는지 보는 기준입니다.

(iii) 통계적 일관성. 같은 변수 $x$에 대해 {평균, 중앙값, 꼬리 폭} 중 최소 두 개가 증거를 보여야 합니다. 하나의 요약 통계에만 의존하는 걸 막습니다.

(iv) 의미적 일관성. 관찰된 분포 변화가 관찰 조건의 텍스트 의미와 맞아야 합니다. 예를 들어 "장중 회복"용 팩터라면 팩터가 강할수록 POS가 높아져야지 낮아지면 안 됩니다. 이 판단은 LLM(검증 에이전트)이 합니다.

**최종 판정**:

$$\mathcal{A}_V(o_i, f_{i,j}, S_{i,j}) = \mathbb{I}\Big[\bigwedge_{c \in K} c(o_i, f_{i,j}, S_{i,j})\Big],\quad \hat{F}_i = \{f_{i,j} \in F_i : \mathcal{A}_V = \text{pass}\}$$

네 기준을 모두(논리곱 $\wedge$) 만족해야 통과입니다. 통과한 팩터만 모은 것이 $\hat{F}_i$이고, 탈락한 팩터는 팩터 에이전트로 돌려보내 최대 5번까지 수정합니다.

### Stage 3: 팩터 통합

**조합 생성**: 관찰 조건마다 통과한 팩터 중 하나씩 골라 조합합니다.

$$\mathcal{C} = \hat{F}_1 \times \hat{F}_2 \times \cdots \times \hat{F}_m,\quad |\mathcal{C}| = \prod_{i=1}^{m} |\hat{F}_i|$$

데카르트 곱이라 조건 3개에 각 2~3개씩이면 8~27개 조합이 나옵니다.

**진입 신호**: 선택도 $\sigma \in (0,1)$을 정하고, 조합 안의 모든 팩터가 각자 학습 기간 $\sigma$-분위수를 극성 방향으로 넘을 때만 신호가 1이 됩니다.

$$\text{Signal}(f, \sigma) = 1 \iff \text{조합 내 모든 팩터가 동시에 } \sigma\text{-분위수 초과}$$

세 조건이 동시에 성립해야 가설이 말하는 시장 상태라는 논리입니다.

**방향적 선택도(Directional Selectivity)**: $\sigma$를 q50 → q70 → q90으로 올려가며 신호를 점점 까다롭게 만듭니다. 가설이 진짜라면 조건이 엄격해질수록, 즉 가설 상태가 더 뚜렷한 경우만 남길수록 성과가 좋아져야 합니다. 구체적으로는 평균 실현 수익률이 오르거나, 승률

$$\frac{|S_W|}{|S_W| + |S_L|}$$

이 올라야 합니다($S_W$는 수익 난 신호, $S_L$은 손실 난 신호). 이걸 종목별로 확인하고, 통과한 종목 비율이 기준(0.5)을 넘는 조합만 남깁니다. 이 단계에서는 수익률을 쓰지만 수익률 크기 자체가 아니라 **엄격할수록 나아지는가**라는 구조를 봅니다.

### Stage 4: 최적화와 백테스트

남은 조합마다 팩터별 진입 임계값을 검증 기간(2024년)에서 Optuna로 20회 탐색합니다. 목적함수는 칼마 비율입니다.

$$\text{Calmar} = \frac{AR}{|MDD|}$$

연수익률을 최대낙폭으로 나눈 값이라 수익과 하방 위험을 함께 봅니다. 정해진 임계값은 고정한 채 2025년 테스트에 적용합니다. 반복 간에는 가설 ID, 관찰 설명, 수식 이름, 평균 IR만 메모리로 넘기고 수식 정의나 임계값은 넘기지 않는다고 명시합니다.

---

## 3. 실험 설정

유니버스는 CSI 500(중국)과 S&P 500(미국), 데이터는 일봉 OHLCV만 쓰며 각각 Baostock과 yfinance에서 받았습니다. 기간은 학습 2022~2023년, 검증 2024년, 테스트 2025년입니다. 백테스트는 Qlib의 TopkDropout(k=50, n=5)을 모든 방법에 동일하게 적용했고, 거래비용은 CSI 500 매수 0.05%·매도 0.15%, S&P 500 매도 0.05%입니다. 기본 LLM은 GPT-4o이고, 비교 대상은 ML/DL(선형, XGBoost, LightGBM, MLP, Transformer), 강화학습(AlphaForge, AlphaQCM), LLM 기반(RD-Agent, AlphaAgent)입니다.

### 평가 지표 (부록 C)

$A_t = R^p_t - R^b_t$를 포트폴리오와 벤치마크의 일간 수익률 차이(초과수익)라 하고, $N$을 연간 거래일 수, $T$를 전체 기간 일수라 합시다.

**연환산 초과수익** $AR = \left(\prod_{t=1}^{T}(1 + A_t)\right)^{N/T} - 1$: 매일의 초과수익을 복리로 쌓은 뒤 1년 기준으로 환산합니다.

**정보비율** $IR = \sqrt{N}\,\dfrac{\mathbb{E}[A_t]}{\text{Std}[A_t]}$: 초과수익 평균을 추적오차로 나눈 값으로, 벤치마크 대비 샤프 비율이라고 보면 됩니다.

**최대낙폭** $DD_t = \dfrac{V_t}{\max_{\tau \le t} V_\tau} - 1,\ \ MDD = \min_t DD_t$: 이건 초과수익이 아니라 포트폴리오 자체 누적가치 $V_t$의 고점 대비 최대 하락입니다.

**누적 초과수익** $CR = \prod_{t=1}^{T}(1 + A_t) - 1$: 연환산하지 않은 전체 기간 초과수익입니다.

---

## 4. 주요 결과

**메인 결과 (Table 1)**: CSI 500에서 AR 0.2067, IR 1.53, MDD −0.085, CR 0.2225로 모든 지표 1위입니다. 2위는 RD-Agent(IR 1.34)입니다. S&P 500에서도 AR 0.1062, IR 1.13으로 1위이고, 2위권은 AlphaQCM(IR 0.83)과 RD-Agent(IR 0.80)입니다. ML/DL 베이스라인은 대부분 음수입니다.

**사례 연구 (Table 2)**: 같은 관찰 조건에서 나온 두 후보가 정반대 판정을 받는 예시를 보여줍니다. "장중 회복"용 팩터 중 하나는 분위가 올라갈수록 POS 평균이 0.083에서 0.914로 단조 증가해 통과하고, 다른 하나는 극성은 "클수록 회복"인데 실제 증거는 반대로 움직여 탈락합니다.

**소거 실험 (Table 4, CSI 500)**: Stage 2를 빼면 IR이 1.53에서 0.15로, Stage 3을 빼면 −1.27로, 둘 다 빼면 −1.43으로 떨어집니다. 저자들은 두 단계가 성과의 존재 자체를 결정한다고 해석합니다.

**LLM 백본 민감도 (Table 3)**: 백본을 바꿔도 AR과 IR은 모두 양수지만, GPT-4o가 압도적이고 나머지는 IR 0.08~1.32 수준으로 편차가 큽니다.

**부록의 추가 분석**: 고정 조합의 2021~2025 연도별 테스트(부록 K)에서 두 시장 모두 매년 양수 초과수익을 냈고, 49개 이동 윈도 중 CSI 500은 33개, S&P 500은 39개가 양수였습니다. 전문가 블라인드 평가(부록 J) 100건에서는 완전 동의 39건, 부분 동의 49건, 불일치 12건이었습니다. 부록 O에서 저자들은 해석 가능성 주장을 "수식과 OHLCV 관찰 조건 사이의 실증적 일관성"으로 스스로 한정하고, 인과적·근본적 경제 메커니즘을 입증하지 않는다고 명시합니다.

---

## 5. 비판적으로 보면

FAMA보다 훨씬 성실한 논문입니다. 테스트를 GPT-4o 지식 시점 이후인 2025년으로 둔 점, 거래비용 반영, 10회 반복 분산 보고, 연도별 고정 조합 테스트, 해석 가능성 주장의 범위 한정까지 FAMA에서 지적했던 문제 상당수를 피했습니다. 그런데도 핵심 주장에는 약점이 있습니다.

**Stage 2 검증이 거의 거르지 않습니다.** 이게 가장 큰 문제입니다. 부록 I에서 첫 반복 수식 461개 중 437개가 통과하고 24개만 탈락했습니다. 통과율이 95%입니다. 구조적으로 그럴 수밖에 없는 이유가 있는데, 팩터는 종가·고가·저가·거래량으로 만들고, 검증은 그 팩터로 나눈 분위에서 같은 날의 종가·고가·저가·거래량 분포가 달라지는지를 봅니다. 같은 재료로 만든 것끼리 비교하니 기계적으로 상관이 생깁니다. 기준 (ii)는 다섯 분위 중 아무 두 개의 꼬리 폭이 $\epsilon$보다 다르기만 하면 통과라 특히 느슨합니다.

**그런데 소거 실험 결과가 너무 큽니다.** 5%만 걸러내는 단계를 뺐는데 IR이 1.53에서 0.15로 무너진다는 건 앞뒤가 잘 안 맞습니다. 게다가 같은 부록 I의 Table 14에서 탈락한 수식들이 8개 비교 중 6개에서 통과 수식보다 RankIC가 더 높습니다. 저자들은 Stage 2가 수익 예측 필터가 아니라 의미 일관성 필터라서 괜찮다고 설명하지만, 그렇다면 Stage 2 제거가 성과를 크게 떨어뜨린 이유는 Stage 2 자체보다 파이프라인 경로가 달라진 데서 온 변동일 가능성을 배제하지 못합니다. IR의 반복 간 표준편차가 0.39라는 점을 감안하면 소거 실험도 반복 평균으로 보고했어야 합니다.

**"결정론적 판정"이라는 설명과 실제 구현이 다릅니다.** 본문 3.2.2절은 LLM이 관찰 조건을 변수에 매핑만 하고 최종 판정은 규칙의 논리곱이라 결정론적이라고 쓰지만, 부록 E의 실제 프롬프트를 보면 LLM이 통계를 읽고 PASS/FAIL과 근거를 직접 출력합니다. 기준 (iv)도 LLM 판단입니다.

**선택 과정의 다중검정 문제가 큽니다.** 부록 L에 따르면 가설 520개, 후보 조합 18,699개를 만들었고 Stage 3을 통과한 반복은 696개 중 49개입니다. 그런데 이 표에 "OOS 수익이 양수인 조합 1,118개"라는 항목이 있다는 건, 많은 조합의 테스트 성과를 실제로 관찰했다는 뜻입니다. 메인 결과에 쓴 가설이 어떻게 선택됐는지, 그 선택이 2025년 성과를 본 뒤에 이뤄진 건 아닌지가 명확하지 않습니다. 조합 수도 부록마다 18,699개, 649개, 212개로 달라 집계 기준이 혼란스럽습니다.

**부록 M이 메인 결과를 약화시킵니다.** Stage 3을 통과한 조합 전체에 공통 선택도 σ=0.55를 적용한 단순 평균 누적 초과수익이 CSI 500에서 0.2048입니다. FaVOR 최종 결과 0.2225와 거의 같습니다. 즉 Optuna로 임계값을 최적화하고 최고 조합을 고르는 과정의 기여는 미미하고, 2025년 CSI 500의 하반기 강세장에서 이런 류의 롱 신호 전반이 잘 작동했을 가능성이 큽니다.

**롱온리 초과수익의 원천을 분해하지 않았습니다.** 벤치마크 대비 초과수익만 보고하고 베타·사이즈·모멘텀 노출로 회귀해서 알파를 분리하지 않았습니다. 선택된 CSI 500 수식(부록 Table 12)이 거래량 급증 비율과 30일 이동평균 대비 종가 위치처럼 단순한 모멘텀·거래량 신호라는 점에서 특히 필요한 분석입니다.

**자기 규칙을 어긴 수식이 최종 선택됐습니다.** 팩터 에이전트 프롬프트는 "가격 수준을 그대로 쓰지 말고 수익률이나 표준화 값을 쓰라"고 지시하는데, S&P 500 최종 조합의 `STD15(close)`는 종가의 표준편차를 달러 단위 그대로 씁니다. 이러면 주가가 높은 종목일수록 값이 커져서, 변동성이 아니라 주가 수준을 측정하게 됩니다.

**"레짐 강건성"은 사후 해석입니다.** 부록 H의 레짐 라벨은 결과 해석용일 뿐 선택이나 거래에 쓰이지 않는다고 명시돼 있습니다. 레짐 강건성의 실제 근거는 부록 K의 연도별 테스트인데, 이건 이 논문에서 가장 설득력 있는 증거이긴 하지만 2021~2023년은 GPT-4o의 학습 시기와 겹쳐 LLM 사전지식 누수를 배제할 수 없습니다. 백본 비교에서도 GPT-5.4-mini나 Claude Sonnet 4.6 같은 최신 모델은 학습 데이터가 2025년을 포함할 가능성이 있어 공정한 비교인지 의문입니다.

**생존편향 언급이 없습니다.** S&P 500 데이터를 yfinance에서 받았는데, 기간 중 지수에서 편출되거나 상장폐지된 종목을 어떻게 처리했는지 설명이 없습니다.

---

## 6. 앞선 연구 방향과의 관계

이 논문은 앞서 말씀드린 방향을 침범하지 않고, 오히려 빈자리를 더 분명하게 보여줍니다.

FaVOR는 레짐을 사후 해석에만 쓰고 마이닝이나 검증에는 넣지 않습니다. 따라서 **레짐 조건부 마이닝**은 여전히 열려 있습니다. 게다가 FaVOR의 Stage 2 구조(팩터 분위별로 같은 날 시장 상태 분포를 비교)는 레짐별로 쪼개 적용하기 쉬운 형태입니다. 예를 들어 "이 팩터는 고변동성 레짐에서만 의도한 상태를 포착하고 저변동성 레짐에서는 의미가 사라진다"를 검증 단계에서 판정하는 식으로 확장할 수 있습니다. 앞서 지적한 Stage 2의 느슨함(같은 날 OHLCV로 같은 날 OHLCV를 검증)을 레짐 분할과 함께 개선하면 방법론 기여도 생깁니다.

한국 시장도 비어 있습니다. FaVOR가 중국과 미국을 다뤘으니 KOSPI/KOSDAQ 적용은 자연스러운 확장이고, 코드가 공개돼 있어(GitHub damilab/FaVOR) 베이스라인으로 바로 재현할 수 있습니다.

실무적으로 하나 더 짚자면, 저자 중 한 명이 숭실대 금융학부 소속입니다. 이 방향으로 논문을 쓰신다면 직접 연락해 의견을 구하거나 지도를 요청해볼 만한 사람이 학내에 있다는 뜻입니다.